In [1]:
import pandas as pd
import pyarrow
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import joblib
import boto3

In [2]:
s3_path = "s3://security-data-lake-1.0/analytics/"

df = pd.read_parquet(s3_path)

print(df.shape)
print(df["Label"].value_counts())

(23675873, 9)
Label
Benign                      20746260
DDoS attacks-LOIC-HTTP       1152382
DDOS attack-HOIC              668461
DoS attacks-Hulk              434873
Bot                           282310
Infilteration                 161059
SSH-Bruteforce                117322
DoS attacks-GoldenEye          41455
FTP-BruteForce                 39346
DoS attacks-SlowHTTPTest       19462
DoS attacks-Slowloris          10285
DDOS attack-LOIC-UDP            1730
Brute Force -Web                 611
Brute Force -XSS                 230
SQL Injection                     87
Name: count, dtype: int64


In [3]:
df["Attack"] = (df["Label"] != "Benign").astype(int)

print(df["Attack"].value_counts())

Attack
0    20746260
1     2929613
Name: count, dtype: int64


In [4]:
df_sample, _ = train_test_split(
    df,
    train_size=1000000,
    stratify=df["Label"],
    random_state=42
)

print(df_sample.shape)
print(df_sample["Attack"].value_counts())

(1000000, 10)
Attack
0    876262
1    123738
Name: count, dtype: int64


In [5]:
features = [
    "Dst Port",
    "Protocol",
    "Flow Duration",
    "Flow Byts/s",
    "Flow Pkts/s",
    "Tot Fwd Pkts",
    "Tot Bwd Pkts"
]

X = df_sample[features]
y = df_sample["Label"]

In [6]:
le = LabelEncoder()

y = le.fit_transform(df_sample["Label"])

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [10]:
xgboost_natural_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective="multi:softmax",
    num_class=len(np.unique(y)),
    random_state=42,
    n_jobs=-1
)

xgboost_natural_model.fit(X_train, y_train)

,objective,'multi:softmax'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [11]:
y_pred = xgboost_natural_model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99    175252
           1       1.00      1.00      1.00      2385
           2       0.00      0.00      0.00         5
           3       0.00      0.00      0.00         2
           4       0.68      0.97      0.80      5647
           5       0.83      0.67      0.74        15
           6       0.98      0.99      0.99      9735
           7       0.85      0.72      0.78       350
           8       0.81      0.36      0.50      3674
           9       0.57      0.46      0.51       164
          10       1.00      0.95      0.98        87
          11       0.75      0.84      0.79       332
          12       0.25      0.00      0.01      1360
          13       0.00      0.00      0.00         1
          14       0.99      1.00      1.00       991

    accuracy                           0.98    200000
   macro avg       0.65      0.60      0.61    200000
weighted avg       0.97   

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"

In [16]:
train_score = xgboost_natural_model.score(X_train, y_train)
test_score = xgboost_natural_model.score(X_test, y_test)

print("Train:", train_score)
print("Test :", test_score)

Train: 0.97563125
Test : 0.97538


In [17]:
joblib.dump(xgboost_natural_model, "xgboost_natural.pkl")

['xgboost_natural.pkl']

In [18]:
joblib.dump(le, "label_encoder.pkl")

['label_encoder.pkl']

In [21]:
bucket = "security-data-lake-1.0"
s3 = boto3.client("s3")

s3.upload_file("xgboost_natural.pkl", bucket, "models/xgboost_natural.pkl")

In [8]:
xgb_model = joblib.load("xgboost_natural.pkl")
le = joblib.load("label_encoder.pkl")

flow = df.iloc[[0]][features]

prediction = xgb_model.predict(flow)
attack_name = le.inverse_transform(prediction)[0]

print("Predicted attack:", attack_name)
print("Actual attack:", df.iloc[0]["Label"])

Predicted attack: Benign
Actual attack: Benign


In [9]:
sample_df = df.sample(20, random_state=42)

flows = sample_df[features]

predictions = xgb_model.predict(flows)
predicted_names = le.inverse_transform(predictions)

results = pd.DataFrame({
    "Actual": sample_df["Label"].values,
    "Predicted": predicted_names
})

print(results)

                    Actual         Predicted
0                   Benign            Benign
1         DDOS attack-HOIC  DDOS attack-HOIC
2                   Benign            Benign
3                   Benign            Benign
4                   Benign            Benign
5                   Benign            Benign
6                   Benign            Benign
7                   Benign            Benign
8                   Benign            Benign
9                   Benign            Benign
10        DDOS attack-HOIC  DDOS attack-HOIC
11                  Benign            Benign
12                  Benign            Benign
13                  Benign            Benign
14                  Benign            Benign
15                  Benign            Benign
16                  Benign            Benign
17  DDoS attacks-LOIC-HTTP            Benign
18                  Benign            Benign
19                  Benign            Benign
